# UrbanEye: Air Pollution AI Model Training

In this notebook, we transform the **Air Pollution Dataset** into a trained Machine Learning model (`scikit-learn` Random Forest). 
The model will predict the **Air Quality Index (AQI)** based on the measured pollutant levels. Once trained and exported, the FastAPI backend will load this model to make real-time predictions for the Interactive Dashboard.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
import joblib
import os

print("Libraries loaded successfully.")

## 1. Data Loading & Preprocessing

In [ ]:
# Define paths
DATA_PATH = r"H:\final year project\Air Pollution Dataset\city_day.csv"
MODEL_EXPORT_PATH = r"H:\final year project\backend\ai_models\pollution_model.pkl"

# Load dataset
df = pd.read_csv(DATA_PATH)
print(f"Loaded dataset with {len(df)} records.")

# The features we will use for prediction
FEATURES = ['PM2.5', 'PM10', 'NO', 'NO2', 'NOx', 'NH3', 'CO', 'SO2', 'O3', 'Benzene', 'Toluene', 'Xylene']
TARGET = 'AQI'

# Clean Data: drop rows where TARGET (AQI) or all features are entirely missing
df_clean = df.dropna(subset=[TARGET])

# For missing feature values, we will fill with the median of that specific column
df_clean[FEATURES] = df_clean[FEATURES].fillna(df_clean[FEATURES].median())

print(f"Data points after cleaning: {len(df_clean)}")

## 2. Model Training

In [ ]:
# Split into training and test sets
X = df_clean[FEATURES]
y = df_clean[TARGET]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training on {len(X_train)} samples, testing on {len(X_test)} samples...")

# Train a Random Forest Regressor
# We limit max_depth and estimators to speed up training on CPU
model = RandomForestRegressor(n_estimators=100, max_depth=15, n_jobs=-1, random_state=42)
model.fit(X_train, y_train)

print("Training Complete!")

## 3. Evaluation

In [ ]:
# Let's see how well it learned
predictions = model.predict(X_test)

mae = mean_absolute_error(y_test, predictions)
r2 = r2_score(y_test, predictions)

print(f"Mean Absolute Error: {mae:.2f} AQI points")
print(f"R-squared Score: {r2:.3f} (closer to 1.0 is better)")

## 4. Export Model for the Backend API

In [ ]:
# Create the directory if it doesn't exist
os.makedirs(os.path.dirname(MODEL_EXPORT_PATH), exist_ok=True)

# Save the trained model to a `.pkl` file so FastAPI can load it
joblib.dump(model, MODEL_EXPORT_PATH)

print(f"✅ Model successfully exported to: {MODEL_EXPORT_PATH}")